# ROGII Wellbore — Colab runner

Generic notebook for running any `notebooks/*.py` training script on Colab Pro with GPU.

**Workflow** (run cells top to bottom):
1. Mount Drive + set Kaggle auth (from Colab Secrets)
2. Clone the repo (latest `main`)
3. **Version meta** — reads `SPRINT_ACTIVE.txt` to pick the active script
4. Install deps (auto-detected from the script's imports)
5. Get competition data (`kaggle competitions download`) + sync artifacts from Drive
6. Run the script
7. Sync artifacts back to Drive
8. Download the newest submission to local

To switch experiments: edit `SPRINT_ACTIVE.txt`, push, then re-run cells **clone → meta → install → run**.


## 1. Mount Drive + Kaggle auth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Colab Notebooks/kaggle/rogii'
# Note the space in 'Colab Notebooks' — Google's auto-created folder name; keep it.
!mkdir -p "$DRIVE_ROOT"
!ls -la "$DRIVE_ROOT" || echo 'Drive folder created (empty).'

# --- Kaggle CLI auth from Colab Secrets (key icon, left sidebar) ---
import os, json
try:
    from google.colab import userdata
    username = userdata.get('KAGGLE_USERNAME')
    api_token = userdata.get('KAGGLE_API_TOKEN')
    if api_token and api_token.strip().startswith('{'):
        parsed = json.loads(api_token)
        api_token = parsed.get('key', api_token)
        username = username or parsed.get('username')
    if not username or not api_token:
        raise ValueError('missing username or token')
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = api_token
    print(f'✓ Kaggle auth set (user: {username})')
except Exception as exc:
    print(f'⚠ Kaggle auth skipped: {exc} — set KAGGLE_USERNAME + KAGGLE_API_TOKEN in Colab Secrets.')


## 2. Clone repo + check out latest

In [ ]:
%cd /content
!rm -rf rogii-wellbore-geology-prediction
!git clone -q https://github.com/SirGrigor/rogii-wellbore-geology-prediction.git
%cd rogii-wellbore-geology-prediction
!git log -1 --oneline


## 3. Version meta — verify which script will run

Edit `SPRINT_ACTIVE.txt` in the repo + push to change this. Re-run the clone cell first.

In [ ]:
import os
SCRIPT = None
EXTRA_ARGS = None
config_path = '/content/rogii-wellbore-geology-prediction/SPRINT_ACTIVE.txt'
if os.path.exists(config_path):
    with open(config_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                parts = line.split(maxsplit=1)
                SCRIPT = parts[0]
                EXTRA_ARGS = parts[1] if len(parts) > 1 else ''
                break
    print(f'✓ Loaded from SPRINT_ACTIVE.txt:  {SCRIPT}  {EXTRA_ARGS}')
else:
    SCRIPT, EXTRA_ARGS = 'notebooks/01_eda.py', ''
    print(f'⚠ SPRINT_ACTIVE.txt not found, defaulted to: {SCRIPT}')

print('=' * 70)
print(f'ABOUT TO RUN:  {SCRIPT}   args: {EXTRA_ARGS or "(none)"}')
print('=' * 70)
!git log -1 --oneline
print(f'--- {SCRIPT} header ---')
!head -30 {SCRIPT}


## 4. Install dependencies (auto-detected from the script's imports)

We do NOT `pip install -e .` (its tool deps are editable local paths that don't exist on Colab). Instead: base deps + the shared **kaggle-playground-utils** from GitHub (public). `src.*` imports work because we run from the cloned repo root. **synth-decoder is local-only** — its GATE/adversarial checks are cheap and run on your machine, not on Colab GPU.

In [ ]:
import os
from pathlib import Path

script_text = Path(SCRIPT).read_text() if SCRIPT and Path(SCRIPT).exists() else ''
s = script_text.lower()

NEEDS_DTW      = 'dtaidistance' in s or 'src.align' in s or 'from src import align' in s
NEEDS_CATBOOST = 'catboost' in s
NEEDS_PYTABKIT = 'pytabkit' in s or 'realmlp' in s or 'tabm' in s
NEEDS_OPTUNA   = 'optuna' in s
NEEDS_TORCH    = 'torch' in s or NEEDS_PYTABKIT

!pip install -q uv

# Base deps + the shared toolkit from GitHub (public). NOT `-e .` — see the note above.
base = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'pyarrow', 'lightgbm', 'xgboost',
        'matplotlib', 'seaborn', 'dtaidistance']
if NEEDS_CATBOOST: base.append('catboost')
if NEEDS_PYTABKIT: base.append('pytabkit')
if NEEDS_OPTUNA:   base.append('optuna')
!uv pip install -q --system {' '.join(base)}
!uv pip install -q --system "git+https://github.com/SirGrigor/kaggle-playground-utils.git"

import lightgbm, sklearn, numpy, pandas
print(f'numpy {numpy.__version__}  pandas {pandas.__version__}  sklearn {sklearn.__version__}  lgb {lightgbm.__version__}')
import kaggle_playground_utils as _kpu; print(f'kaggle-playground-utils {_kpu.__version__}  (diary/observer/viz available)')
if NEEDS_TORCH:
    import torch
    print(f'torch {torch.__version__}  CUDA={torch.cuda.is_available()}  '
          f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## 5. Get competition data + sync prior artifacts from Drive

In [ ]:
import os, shutil

for d in ('data/raw', 'data/external', 'data/splits', 'probs', 'submissions', 'reports'):
    os.makedirs(d, exist_ok=True)

# Competition data: download once per session (many small per-well CSVs → use the CLI).
if not os.listdir('data/raw') or not os.path.isdir('data/raw/train'):
    print('Downloading competition data...')
    !kaggle competitions download -c rogii-wellbore-geology-prediction -p data/raw
    !cd data/raw && (ls *.zip >/dev/null 2>&1 && unzip -o -q *.zip || echo 'no zip to unzip')
else:
    print('data/raw already populated.')

# Prior artifacts from Drive (probs / submissions / experiments diary)
for sub in ('probs', 'submissions'):
    src_root = f'/content/drive/MyDrive/Colab Notebooks/kaggle/rogii/{sub}'
    if os.path.isdir(src_root):
        for name in sorted(os.listdir(src_root)):
            s, d = f'{src_root}/{name}', f'{sub}/{name}'
            if os.path.isdir(s):
                os.makedirs(d, exist_ok=True)
                for fn in os.listdir(s):
                    if not os.path.exists(f'{d}/{fn}'):
                        shutil.copyfile(f'{s}/{fn}', f'{d}/{fn}')
            elif not os.path.exists(d):
                shutil.copyfile(s, d)
        print(f'  synced {sub}/ from Drive')

print('--- data/raw ---')
!ls data/raw | head


## 6. Run the target script

In [ ]:
print(f'RUNNING: {SCRIPT}  {EXTRA_ARGS}')
import os; print('cwd:', os.getcwd())
!python {SCRIPT} {EXTRA_ARGS}


## 7. Sync artifacts → Drive

In [ ]:
import os, shutil
synced = False
for sub in ('probs', 'submissions'):
    if os.path.isdir(sub):
        for name in sorted(os.listdir(sub)):
            s = f'{sub}/{name}'
            d = f'/content/drive/MyDrive/Colab Notebooks/kaggle/rogii/{sub}/{name}'
            if os.path.isdir(s):
                os.makedirs(d, exist_ok=True)
                for fn in os.listdir(s):
                    shutil.copyfile(f'{s}/{fn}', f'{d}/{fn}')
            else:
                os.makedirs(os.path.dirname(d), exist_ok=True)
                shutil.copyfile(s, d)
        print(f'  synced {sub}/ → Drive'); synced = True
if os.path.exists('experiments.jsonl'):
    shutil.copyfile('experiments.jsonl', f'/content/drive/MyDrive/Colab Notebooks/kaggle/rogii/experiments.jsonl')
    print('  synced experiments.jsonl → Drive'); synced = True
print('✓ sync complete.' if synced else 'ABORT: nothing to sync — did the run cell fail?')


## 8. Download newest submission to local

In [ ]:
from pathlib import Path
from google.colab import files
subs = sorted(Path('submissions').glob('*.csv'), key=lambda p: -p.stat().st_mtime)
if not subs:
    print('⚠ No submissions — the run cell probably failed.')
else:
    newest = subs[0]
    print(f'Downloading {newest.name}. Submit from local with:')
    print(f'  kaggle competitions submit -c rogii-wellbore-geology-prediction -f submissions/{newest.name} -m "from {SCRIPT}"')
    files.download(str(newest))
